# Style adapters: dataset and training

An experiment notebook: it only needs the libraries. The code of the project (`data/` and `src/`) is copied here, the parameters are plain variables, no configs, no resume.

1. build the captioned WikiArt subset;
2. the dataset: load it and look at the data;
3. train an adapter on it;
4. look at the samples.

Run the cells from top to bottom. To start another run restart the kernel (the models stay in the memory). All the paths are relative to the working directory.

In [ ]:
import hashlib
import json
import math
import re
import shutil
from collections.abc import Callable, Iterable, Mapping
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
from accelerate import Accelerator
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler
from diffusers.training_utils import cast_training_params, set_seed
from IPython.display import display
from peft import LoraConfig, PeftConfig, get_peft_model_state_dict
from PIL import Image
from safetensors.torch import save_file
from torch import nn
from torch.utils.data import DataLoader
from torchvision.transforms import InterpolationMode, v2
from tqdm.auto import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    CLIPTextModel,
    CLIPTokenizer,
    PreTrainedTokenizerBase,
)

ROOT = Path.cwd()
print(ROOT)

## 1. Dataset building

Subset of WikiArt: `TRAIN_SIZE` + `TEST_SIZE` paintings of every artist, captioned by a VLM (needs a GPU), the trigger word of the artist is added to every caption. If the dataset is already built, it is not built again.

### Code: `data/captions.py`, `data/wikiart.py`

In [ ]:
_LEADING = re.compile(
    r'^(the|this|in this)\s+(image|picture|painting|artwork|scene|illustration)\s*'
    r'(shows|depicts|features|portrays|presents|captures|is of|,)?\s*',
    re.IGNORECASE,
)


_LEAK = re.compile(
    r'\b(paint\w*|artwork|artist|style|brush\w*|oil|canvas|impression\w*|realis\w*|romantic\w*|'
    r'art nouveau|symbolis\w*|watercolou?r)\b',
    re.IGNORECASE,
)


def _clean_caption(text: str) -> str:
    c = text.strip().strip('"\'').strip()
    c = _LEADING.sub('', c)
    c = re.sub(r'\s+', ' ', c).strip()
    if len(c) > 1 and c[0].isupper() and not c[1].isupper():
        c = c[0].lower() + c[1:]
    return c.rstrip('. ')


def _needs_review(caption: str, artist_slug: str) -> bool:
    surname = artist_slug.split('-')[-1]
    return bool(_LEAK.search(caption)) or surname.lower() in caption.lower() or len(caption) < 10


def resolve_vlm(
    model_name: str | None,
    batch_size: int | None,
    large_model: str,
    small_model: str,
    large_vram_gb: float,
    large_batch_size: int,
    small_batch_size: int,
) -> tuple[str, int]:
    if not torch.cuda.is_available():
        raise RuntimeError('No GPU found. Please enable a GPU.')

    vram = torch.cuda.get_device_properties(0).total_memory / 2**30
    large = vram >= large_vram_gb
    model_name = model_name or (large_model if large else small_model)
    batch_size = batch_size or (large_batch_size if large else small_batch_size)
    print(
        f'GPU: {torch.cuda.get_device_name(0)}, {vram:.0f} GB -> {model_name}, batch {batch_size}'
    )
    return model_name, batch_size


def load_vlm(model_name: str, device: str) -> tuple[AutoProcessor, AutoModelForImageTextToText]:
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    processor = AutoProcessor.from_pretrained(model_name)
    processor.tokenizer.padding_side = 'left'
    model = AutoModelForImageTextToText.from_pretrained(
        model_name,
        dtype=dtype,
        device_map=device,
    ).eval()
    return processor, model


def add_captions(
    ds: DatasetDict,
    device: str,
    model_name: str,
    batch_size: int,
    prompt: str,
    max_new_tokens: int,
    max_side: int,
) -> DatasetDict:
    processor, model = load_vlm(model_name, device)
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}]}]
    chat_text = processor.apply_chat_template(  # ty:ignore[unresolved-attribute]
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    @torch.inference_mode()
    def caption_batch(batch: dict[str, list]) -> dict[str, list]:
        thumbs = []
        for im in batch['image']:
            im = im.convert('RGB')
            im.thumbnail((max_side, max_side))
            thumbs.append(im)
        inputs = processor(  # ty:ignore[call-non-callable]
            text=[chat_text] * len(thumbs),
            images=thumbs,
            padding=True,
            return_tensors='pt',
        ).to(device)
        out = model.generate(  # noqa: F821 # ty:ignore[unresolved-attribute]
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
        raw = processor.batch_decode(  # ty:ignore[unresolved-attribute]
            out[:, inputs['input_ids'].shape[1] :],
            skip_special_tokens=True,
        )
        caps = [_clean_caption(r) for r in raw]
        review = [_needs_review(c, s) for c, s in zip(caps, batch['artist_slug'])]
        return {'caption': caps, 'caption_raw': raw, 'needs_review': review}

    prompt_hash = hashlib.md5(f'{prompt}|{max_new_tokens}|{max_side}'.encode()).hexdigest()[:8]
    model_tag = model_name.replace('/', '_')
    result = DatasetDict(
        {
            name: split.map(
                caption_batch,
                batched=True,
                batch_size=batch_size,
                desc=f'captioning {name}',
                new_fingerprint=f'cap-{model_tag}-{prompt_hash}-{name}-{len(split)}',
            )
            for name, split in ds.items()
        }
    )
    del model
    torch.cuda.empty_cache()
    return result

In [ ]:
def get_data_dir(
    root: str | Path,
    artists: Mapping[str, str],
    train_size: int,
    test_size: int,
    seed: int,
    target: str,
) -> Path:
    """Directory of the captioned dataset; the name encodes everything that defines its content."""
    # triggers are baked into the prompts, so they are part of the dataset identity
    key = ','.join(f'{slug}:{trigger}' for slug, trigger in sorted(artists.items()))
    artists_hash = hashlib.md5(key.encode()).hexdigest()[:6]
    name = f'wikiart_{artists_hash}_train{train_size}_test{test_size}_seed{seed}_{target}'
    return Path(root) / name


def build_subset(
    ds: Dataset,
    artists: Iterable[str],
    train_size: int = 50,
    test_size: int = 10,
    seed: int = 42,
) -> DatasetDict:
    names = ds.features['artist'].names
    artist_col = np.array(list(ds['artist']))  # labels only, images are not decoded

    rng = np.random.default_rng(seed)
    test_rng = np.random.default_rng([seed, 1])
    train_indices, test_indices = [], []
    for slug in artists:
        if slug not in names:
            raise ValueError(f"Artist '{slug}' not found in the dataset.")
        idx = np.flatnonzero(artist_col == names.index(slug))
        k = min(train_size, len(idx))
        if k < train_size:
            print(f'[warn] {slug} has only {len(idx)} paintings, taking all of them for train')
        train_idx = rng.choice(idx, size=k, replace=False)
        rest = np.setdiff1d(idx, train_idx)
        k_test = min(test_size, len(rest))
        if k_test < test_size:
            print(f'[warn] {slug}: only {k_test} paintings left for test instead of {test_size}')
        test_idx = test_rng.choice(rest, size=k_test, replace=False)
        train_indices.extend(train_idx.tolist())
        test_indices.extend(test_idx.tolist())

    feats = ds.features

    def labels_to_str(batch: dict[str, list]) -> dict[str, list[str]]:
        out = {'artist_slug': [feats['artist'].int2str(x) for x in batch['artist']]}
        for col in ('genre', 'style'):
            if col in batch:
                out[f'{col}_name'] = [feats[col].int2str(x) for x in batch[col]]
        return out

    subset = DatasetDict(
        {
            'train': ds.select(sorted(train_indices)),
            'test': ds.select(sorted(test_indices)),
        }
    )
    return subset.map(labels_to_str, batched=True, desc='labels -> str')


def add_prompts(sub: DatasetDict, artists: Mapping[str, str]) -> DatasetDict:
    def fn(slugs: list[str], captions: list[str]) -> dict[str, list[str]]:
        triggers = [artists[s] for s in slugs]
        return {'trigger': triggers, 'prompt': [f'{t}, {c}' for t, c in zip(triggers, captions)]}

    return sub.map(
        fn,
        batched=True,
        input_columns=['artist_slug', 'caption'],
        desc='trigger -> prompt',
    )


def build_dataset(
    data_dir: str | Path,
    artists: Mapping[str, str],
    train_size: int,
    test_size: int,
    seed: int,
    model_name: str,
    batch_size: int,
    prompt: str,
    max_new_tokens: int,
    max_side: int,
    device: str = 'cuda',
) -> DatasetDict:
    data_dir = Path(data_dir)
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    ds = load_dataset('huggan/wikiart', split='train')
    sub = build_subset(ds, artists, train_size, test_size, seed)
    sub = add_captions(sub, device, model_name, batch_size, prompt, max_new_tokens, max_side)
    sub = add_prompts(sub, artists)

    tmp = data_dir.with_name(data_dir.name + '_tmp')
    if tmp.is_dir():
        shutil.rmtree(tmp)
    sub.save_to_disk(tmp)
    if data_dir.is_dir():
        shutil.rmtree(data_dir)
    tmp.rename(data_dir)

    data = load_from_disk(data_dir)
    assert isinstance(data, DatasetDict)
    for name, split in data.items():
        split.remove_columns('image').to_csv(data_dir / f'captions_{name}.csv')
    num_rev = sum(sum(split['needs_review']) for split in data.values())
    sizes = ', '.join(f'{len(split)} {name}' for name, split in data.items())
    print(f'Saved {sizes} examples to {data_dir}; flagged for review: {num_rev}')
    return data

### Build

In [ ]:
DATA_ROOT = 'data/train_data/wikiart/raw'
ARTISTS = {  # wikiart slug -> trigger word
    'ivan-aivazovsky': 'aivazovsky style',
    'ivan-shishkin': 'shishkin style',
    'isaac-levitan': 'levitan style',
    'arkhip-kuindzhi': 'kuindzhi style',
    'mikhail-vrubel': 'vrubel style',
    'ivan-bilibin': 'bilibin style',
    'ilya-repin': 'repin style',
    'viktor-vasnetsov': 'vasnetsov style',
}
TRAIN_SIZE = 50  # paintings per artist
TEST_SIZE = 10  # held-out paintings per artist
DATA_SEED = 42
CAPTION_TARGET = 'clip'  # clip -> SD 1.5 / SD 2 / SDXL (short captions)

CAPTION_RULES = (
    'Do NOT mention the artist, the art style or movement, the medium, the technique or brushwork, '
    'and do not say that this is a painting or an artwork. '
    'If the image is a portrait, a sketch, a study, or has a decorative border or text, say so. '
    "Do not start with phrases like 'The image shows' or 'This painting depicts'. "
    'Write in English, plain prose, no lists, no quotes.'
)
CAPTION_PROMPT = (
    'Write a short caption describing the content of this image, for training a text-to-image model. '
    'Mention the main subjects, the setting and the composition; mention time of day, weather or '
    f'lighting only if notable. {CAPTION_RULES} One sentence, at most 30 words.'
)

DATA_DIR = get_data_dir(DATA_ROOT, ARTISTS, TRAIN_SIZE, TEST_SIZE, DATA_SEED, CAPTION_TARGET)
if DATA_DIR.is_dir():
    print(f'The dataset is ready: {DATA_DIR}')
else:
    vlm_name, vlm_batch_size = resolve_vlm(  # the VLM is picked by the VRAM size
        model_name=None,
        batch_size=None,
        large_model='Qwen/Qwen2.5-VL-7B-Instruct',
        small_model='Qwen/Qwen2.5-VL-3B-Instruct',
        large_vram_gb=22,
        large_batch_size=8,
        small_batch_size=4,
    )
    build_dataset(
        data_dir=DATA_DIR,
        artists=ARTISTS,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        seed=DATA_SEED,
        model_name=vlm_name,
        batch_size=vlm_batch_size,
        prompt=CAPTION_PROMPT,
        max_new_tokens=60,
        max_side=768,
    )

## 2. Dataset

The code of the dataset is copied from `src/dataset` (`get_root()` is the variable `ROOT`), then a look at the data.

In [ ]:
def load_artist_split(
    path: str | Path,
    artists: Iterable[str] | None = None,
    split: str = 'train',
) -> Dataset:
    data = load_from_disk(ROOT / path)[split]
    if artists is not None:
        keep = set(artists)
        data = data.filter(lambda slug: slug in keep, input_columns='artist_slug')  # ty:ignore[unresolved-attribute]
        if missing := keep - set(data.unique('artist_slug')):
            raise ValueError(f"No paintings of {sorted(missing)} in the '{split}' split of {path}.")
    print(f'{len(data)} paintings of {sorted(artists) if artists else "all artists"} ({split})')
    return data  # ty:ignore[invalid-return-type]


def save_holdout(data: Dataset, save_path: Path) -> list[str]:
    save_path.mkdir(parents=True, exist_ok=True)
    prompts = []
    for j, row in enumerate(data):
        row['image'].convert('RGB').save(save_path / f'{j:02d}.jpg', quality=95)
        prompts.append(row['prompt'])
    write_json(prompts, save_path / 'prompts.json')
    return prompts

In [ ]:
class StyleDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        path: str,
        artists: list[str] | None,
        tokenizer: PreTrainedTokenizerBase,
        split: str = 'train',
        resolution: int = 512,
        center_crop: bool = True,
        random_flip: bool = False,
    ) -> None:
        self.data = load_artist_split(path, artists, split)
        self.tokenizer = tokenizer
        crop = v2.CenterCrop(resolution) if center_crop else v2.RandomCrop(resolution)
        self.transform = v2.Compose(
            [
                v2.Resize(resolution, interpolation=InterpolationMode.LANCZOS),
                crop,
                *([v2.RandomHorizontalFlip()] if random_flip else []),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize([0.5], [0.5]),
            ]
        )

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        row = self.data[index]
        input_ids = self.tokenizer(
            row['prompt'],
            max_length=self.tokenizer.model_max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids[0]
        pixel_values = self.transform(row['image'].convert('RGB'))
        return {'pixel_values': pixel_values, 'input_ids': input_ids}

    def __len__(self) -> int:
        return len(self.data)

In [ ]:
samples = load_artist_split(DATA_DIR, split='train').shuffle(seed=0).select(range(4))
for row in samples:
    print(row['prompt'])
display(
    Image.fromarray(
        np.hstack([np.array(row['image'].convert('RGB').resize((256, 256))) for row in samples])
    )
)

## 3. Training

First the code of the training (copied from `src/`), then the run.

### Code: `src/utils`, `src/model`, `src/training`

In [ ]:
def read_json(fname: str | Path) -> list[dict[str, Any]] | dict[str, Any]:
    """
    Read the given json file.

    Parameters
    ----------
    fname : str or Path
        Filename of the json file.

    Returns
    -------
    list[dict[str, Any]] or dict[str, Any]
        Loaded json.
    """
    fname = Path(fname)
    with fname.open('rt') as handle:
        return json.load(handle, object_hook=dict)


def write_json(content: Any, fname: str | Path) -> None:
    """
    Write the content to the given json file.

    Parameters
    ----------
    content : Any
        Content to write (must be JSON-friendly).
    fname : str or Path
        Filename of the json file.
    """
    fname = Path(fname)
    with fname.open('wt') as handle:
        json.dump(content, handle, indent=4, sort_keys=False)

In [ ]:
def get_parameter_summary(model: nn.Module) -> str:
    """Return formatted model parameter counts."""

    def format_number(num: int) -> str:
        if abs(num) >= 1_000_000:
            return f'{num / 1_000_000:.2f}M'
        elif abs(num) >= 1_000:
            return f'{num / 1_000:.2f}K'
        else:
            return str(num)

    all_parameters = sum([p.numel() for p in model.parameters()])
    trainable_parameters = sum([p.numel() for p in model.parameters() if p.requires_grad])

    return (
        f'All parameters: {format_number(all_parameters)}\n'
        f'Trainable parameters: {format_number(trainable_parameters)}'
    )

In [ ]:
@dataclass
class SDComponents:
    tokenizer: CLIPTokenizer
    text_encoder: CLIPTextModel
    vae: AutoencoderKL
    unet: UNet2DConditionModel
    noise_scheduler: DDPMScheduler


DTYPES = {
    'no': torch.float32,  # the name of the accelerator's mixed precision
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16,
}


def load_sd(model_name: str, device: torch.device | str, dtype: str = 'fp32') -> SDComponents:
    if dtype not in DTYPES:
        raise ValueError(f"Unknown dtype '{dtype}', choose one of {list(DTYPES)}.")
    _dtype = DTYPES[dtype]
    sd = SDComponents(
        tokenizer=CLIPTokenizer.from_pretrained(model_name, subfolder='tokenizer'),
        text_encoder=CLIPTextModel.from_pretrained(
            model_name, subfolder='text_encoder', dtype=_dtype
        ),
        vae=AutoencoderKL.from_pretrained(model_name, subfolder='vae', dtype=_dtype),
        unet=UNet2DConditionModel.from_pretrained(model_name, subfolder='unet', dtype=_dtype),
        noise_scheduler=DDPMScheduler.from_pretrained(model_name, subfolder='scheduler'),  # ty:ignore[ invalid-argument-type]
    )
    for m in (sd.text_encoder, sd.vae, sd.unet):
        m.requires_grad_(False)  # ty:ignore[unresolved-attribute]
        m.to(device)  # ty:ignore[invalid-argument-type,unresolved-attribute]
    return sd


def attach_adapter(
    unet: UNet2DConditionModel,
    peft_config: PeftConfig,
    gradient_checkpointing: bool = True,
) -> None:
    unet.add_adapter(peft_config)  # ty:ignore[unresolved-attribute]
    cast_training_params(unet, dtype=torch.float32)  # ty:ignore[invalid-argument-type]
    if gradient_checkpointing:
        unet.enable_gradient_checkpointing()  # ty:ignore[unresolved-attribute]

In [ ]:
def save_adapter(unet: UNet2DConditionModel, peft_config: PeftConfig, save_path: Path) -> None:
    save_path.mkdir(parents=True, exist_ok=True)
    peft_config.save_pretrained(str(save_path))
    state_dict = {k: v.contiguous() for k, v in get_peft_model_state_dict(unet).items()}
    save_file(state_dict, save_path / 'adapter_model.safetensors')

In [ ]:
def get_val_prompts(
    templates: list[str],
    holdout: Dataset,
    num_holdout: int,
    out_dir: Path,
) -> list[str]:
    """
    Prompts for the samples: the templates and the captions of the held-out paintings.

    A template with {trigger} is used once per artist (trigger of the artist), the others once.
    num_holdout paintings of every artist are taken. The held-out paintings and the prompts
    are saved to out_dir for comparison with the samples.
    """
    triggers = list(dict.fromkeys(holdout['trigger']))  # unique, in the order of appearance
    prompts = []
    for template in templates:
        if '{trigger}' in template:
            prompts.extend(template.format(trigger=trigger) for trigger in triggers)
        else:
            prompts.append(template)

    counts: dict[str, int] = {}
    indices = []
    for i, slug in enumerate(holdout['artist_slug']):
        if counts.get(slug, 0) < num_holdout:
            indices.append(i)
            counts[slug] = counts.get(slug, 0) + 1
    prompts += save_holdout(holdout.select(indices), out_dir / 'holdout')

    write_json(prompts, out_dir / 'prompts.json')
    return prompts


def build_pipeline(
    sd: SDComponents, unet: UNet2DConditionModel, device: torch.device
) -> StableDiffusionPipeline:
    """Pipeline that shares the weights with the unet that is trained."""
    pipe = StableDiffusionPipeline(
        vae=sd.vae,
        text_encoder=sd.text_encoder,
        tokenizer=sd.tokenizer,
        unet=unet,
        scheduler=DPMSolverMultistepScheduler.from_config(sd.noise_scheduler.config),
        safety_checker=None,
        feature_extractor=None,
        requires_safety_checker=False,
    )
    pipe.to(device)  # no dtype: the LoRA weights stay in fp32
    pipe.set_progress_bar_config(disable=True)
    return pipe


@torch.no_grad()
def generate_samples(
    pipe: StableDiffusionPipeline,
    accelerator: Accelerator,
    prompts: list[str],
    out_dir: Path,
    step: int,
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
    seed: int = 42,
) -> None:
    """Generate images for the fixed prompts with the fixed noise: runs are comparable by step."""
    was_training = pipe.unet.training
    pipe.unet.eval()
    step_dir = out_dir / f'step_{step:04d}'
    step_dir.mkdir(parents=True, exist_ok=True)
    for i, prompt in enumerate(prompts):
        with accelerator.autocast():
            image = pipe(
                prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=torch.Generator('cpu').manual_seed(seed),  # same noise on any device
            ).images[0]
        image.save(step_dir / f'{i:02d}.png')
    pipe.unet.train(was_training)


class ValMetrics:
    """
    Cheap validation metrics: no image generation, only passes of the unet.

    The loss is measured on held-out paintings of the same artists with the fixed noise and
    timesteps, so it is comparable between the steps. It is measured with the caption of the
    painting with the trigger and without it. The adapter is initialized to the identity, so
    the losses at the creation are those of the base model: the gains are measured from them.

    - val/loss, val/loss_no_trigger: MSE of the noise (or velocity) prediction.
    - val/gain, val/gain_no_trigger: the loss of the base model minus the loss now, the higher
      the better. If the gain without the trigger is about the gain with it, the style leaks to
      the prompts without the trigger.
    - adapter/delta_norm: the distance of the trainable weights from their initialization,
      it is 0 if the adapter does not train.
    """

    def __init__(
        self,
        sd: SDComponents,
        accelerator: Accelerator,
        path: str,
        artists: list[str] | None,
        resolution: int,
        images_per_artist: int = 4,
        num_timesteps: int = 5,
        batch_size: int = 8,
        seed: int = 42,
    ) -> None:
        self.sd, self.accelerator, self.batch_size = sd, accelerator, batch_size
        device = accelerator.device
        # the same preprocessing as in the training; the prompt with the trigger is tokenized there
        data = StyleDataset(path, artists, sd.tokenizer, 'test', resolution, True, False)
        counts: dict[str, int] = {}
        indices = []
        for i, slug in enumerate(data.data['artist_slug']):
            if counts.get(slug, 0) < images_per_artist:
                indices.append(i)
                counts[slug] = counts.get(slug, 0) + 1
        rows = [data[i] for i in indices]
        ids = {
            'trigger': torch.stack([r['input_ids'] for r in rows]),
            'plain': sd.tokenizer(
                [data.data[i]['caption'] for i in indices],
                max_length=sd.tokenizer.model_max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            ).input_ids,
        }
        pixels = torch.stack([r['pixel_values'] for r in rows])

        # the vae and the text encoder are frozen: encode once, only the unet runs later
        with torch.no_grad(), accelerator.autocast():
            self.latents = (
                torch.cat(
                    [
                        sd.vae.encode(
                            pixels[i : i + batch_size].to(device, sd.vae.dtype)
                        ).latent_dist.mean.float()
                        for i in range(0, len(pixels), batch_size)
                    ]
                )
                * sd.vae.config.scaling_factor
            )
            self.embeds = {
                name: torch.cat(
                    [
                        sd.text_encoder(x[i : i + batch_size].to(device))[0]
                        for i in range(0, len(x), batch_size)
                    ]
                )
                for name, x in ids.items()
            }
        num_train = sd.noise_scheduler.config.num_train_timesteps
        self.timesteps = torch.linspace(0, num_train - 1, num_timesteps + 2)[1:-1].long().tolist()
        generator = torch.Generator('cpu').manual_seed(seed)
        noise_shape = (num_timesteps, len(rows), *self.latents.shape[1:])
        self.noise = torch.randn(noise_shape, generator=generator).to(device)

        self.params = [p for p in accelerator.unwrap_model(sd.unet).parameters() if p.requires_grad]
        self.init_params = [p.detach().float().clone() for p in self.params]
        self.base = self._losses()
        accelerator.print(
            f'Validation loss of the base model: {self.base["trigger"]:.4f} with the trigger, '
            f'{self.base["plain"]:.4f} without it ({len(rows)} paintings, '
            f'timesteps {self.timesteps})'
        )

    @torch.no_grad()
    def _losses(self) -> dict[str, float]:
        scheduler = self.sd.noise_scheduler
        unet = self.accelerator.unwrap_model(self.sd.unet)
        was_training = unet.training
        unet.eval()
        device = self.accelerator.device
        losses = {}
        for name, embeds in self.embeds.items():
            total = 0.0
            for k, t in enumerate(self.timesteps):
                for i in range(0, len(self.latents), self.batch_size):
                    latents = self.latents[i : i + self.batch_size]
                    noise = self.noise[k, i : i + self.batch_size]
                    timesteps = torch.full((len(latents),), t, device=device, dtype=torch.long)
                    noisy = scheduler.add_noise(latents, noise, timesteps)
                    with self.accelerator.autocast():
                        pred = unet(noisy, timesteps, embeds[i : i + self.batch_size]).sample
                    if scheduler.config.prediction_type == 'v_prediction':
                        target = scheduler.get_velocity(latents, noise, timesteps)
                    else:
                        target = noise
                    total += F.mse_loss(pred.float(), target.float(), reduction='sum').item()
            losses[name] = total / (len(self.timesteps) * self.noise[0].numel())
        unet.train(was_training)
        return losses

    @torch.no_grad()
    def __call__(self) -> dict[str, float]:
        """Metrics of the current state of the adapter, to log with accelerator.log."""
        now = self._losses()
        delta = sum(
            ((p.detach().float() - p0) ** 2).sum().item()
            for p, p0 in zip(self.params, self.init_params)
        )
        return {
            'val/loss': now['trigger'],
            'val/loss_no_trigger': now['plain'],
            'val/gain': self.base['trigger'] - now['trigger'],
            'val/gain_no_trigger': self.base['plain'] - now['plain'],
            'adapter/delta_norm': delta**0.5,
        }

In [ ]:
def get_mixed_precision(mixed_precision: str | None = None) -> str:
    """Mixed precision for the accelerator; None -> bf16 / fp16 on CUDA, no otherwise."""
    if mixed_precision is not None:
        return mixed_precision
    if not torch.cuda.is_available():
        return 'no'
    return 'bf16' if torch.cuda.is_bf16_supported() else 'fp16'


def get_steps_per_epoch(train_loader: DataLoader, grad_accum: int, num_processes: int = 1) -> int:
    """
    Optimizer steps in one epoch.

    The last incomplete accumulation cycle of an epoch is also a step (accelerate syncs the
    gradients at the end of the dataloader). Pass num_processes for a loader that is not prepared.
    """
    return math.ceil(math.ceil(len(train_loader) / num_processes) / grad_accum)  # ty:ignore[invalid-argument-type]


def train(
    accelerator: Accelerator,
    sd: SDComponents,
    optimizer: torch.optim.Optimizer,
    lr_scheduler: Any,
    train_loader: DataLoader,
    num_epochs: int,
    max_grad_norm: float = 1.0,
    log_step: int = 10,
    eval_every: int = 100,
    eval_fn: Callable[[int], None] | None = None,
    save_every: int = 500,
    save_fn: Callable[[int], None] | None = None,
) -> None:
    """
    Train the adapter of sd.unet for num_epochs epochs over the train loader.

    Parameters
    ----------
    sd : SDComponents
        Frozen vae, text encoder and noise scheduler, and the unet with the adapter: it must be
        the one after accelerator.prepare (sd.unet = accelerator.prepare(sd.unet)).
    train_loader : DataLoader
        Train loader after accelerator.prepare.
    eval_fn : Callable[[int], None], optional
        Called with the step number every eval_every steps and at the last step, on all processes.
    save_fn : Callable[[int], None], optional
        The same for saving the adapter: every save_every steps and at the last step.
    """
    device = accelerator.device
    unet = sd.unet
    unet.train()
    params = [p for p in unet.parameters() if p.requires_grad]
    steps_per_epoch = get_steps_per_epoch(train_loader, accelerator.gradient_accumulation_steps)
    num_steps = num_epochs * steps_per_epoch
    step, loss_sum, grad_norm = 0, 0.0, None
    pbar = tqdm(
        total=num_steps,
        desc='train',
        disable=not accelerator.is_local_main_process,
    )
    for _ in range(num_epochs):
        for batch in train_loader:
            with accelerator.accumulate(unet):
                pixel_values = batch['pixel_values'].to(device, dtype=sd.vae.dtype)
                input_ids = batch['input_ids'].to(device)

                with torch.no_grad():
                    latents = sd.vae.encode(pixel_values).latent_dist.sample()
                    latents = latents * sd.vae.config.scaling_factor
                    noise = torch.randn_like(latents)
                    timesteps = torch.randint(
                        0, sd.noise_scheduler.config.num_train_timesteps, (latents.shape[0],)
                    ).to(device)
                    noisy_latents = sd.noise_scheduler.add_noise(latents, noise, timesteps)
                    with accelerator.autocast():
                        text_embeds = sd.text_encoder(input_ids)[0]

                if sd.noise_scheduler.config.prediction_type == 'v_prediction':
                    target = sd.noise_scheduler.get_velocity(latents, noise, timesteps)
                else:
                    target = noise

                pred = unet(noisy_latents, timesteps, text_embeds).sample
                loss = F.mse_loss(pred.float(), target.float())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    grad_norm = accelerator.clip_grad_norm_(params, max_grad_norm)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            loss_sum += loss.item() / accelerator.gradient_accumulation_steps
            if accelerator.sync_gradients:  # a full accumulation cycle is done
                step += 1
                pbar.update(1)
                if step % log_step == 0:
                    lr = lr_scheduler.get_last_lr()[0]
                    pbar.set_postfix(loss=loss_sum / log_step, lr=lr)
                    accelerator.print(f'step {step}: loss={loss_sum / log_step:.4f}, lr={lr:.2e}')
                    metrics = {'train/loss': loss_sum / log_step, 'train/lr': lr}
                    if grad_norm is not None:  # the norm before the clipping
                        metrics['train/grad_norm'] = float(grad_norm)
                    accelerator.log(metrics, step=step)
                    loss_sum = 0.0
                last = step == num_steps
                if eval_fn is not None and (step % eval_every == 0 or last):
                    eval_fn(step)
                    accelerator.wait_for_everyone()
                if save_fn is not None and (step % save_every == 0 or last):
                    save_fn(step)
                    accelerator.wait_for_everyone()
    pbar.close()

### Run

The parameters of the run. Everything defined in the cell below is saved to `params.json` of the run. `RUN_NAME` is the directory `logs/art-style-adapters/{RUN_NAME}` with the checkpoints (the adapters) and the samples.

In [ ]:
_defined = set(globals())

MODEL = 'stable-diffusion-v1-5/stable-diffusion-v1-5'  # or sd2-community/stable-diffusion-2-1
RESOLUTION = 512  # 768 for SD 2.1
TRAIN_ARTISTS = None  # e.g. ['ivan-aivazovsky']; None -> all artists of the dataset
CENTER_CROP = True  # False -> random crop
RANDOM_FLIP = False
SEED = 42
MIXED_PRECISION = None  # None -> bf16 / fp16 on CUDA, no otherwise; or 'no' / 'fp16' / 'bf16'

BATCH_SIZE = 1
NUM_WORKERS = 2
GRAD_ACCUM = 4
NUM_EPOCHS = 20  # epochs over the train loader
LR = 1e-4
WEIGHT_DECAY = 1e-2
WARMUP_STEPS = 100
MAX_GRAD_NORM = 1.0
GRADIENT_CHECKPOINTING = True

LORA_RANK = 16
LORA_ALPHA = 16
TARGET_MODULES = ['to_q', 'to_k', 'to_v', 'to_out.0']

LOG_STEP = 10  # optimizer steps
SAVE_EVERY = 500  # optimizer steps; the adapter is also saved at the last step
VAL_EVERY = 100  # optimizer steps; samples are also generated before and after the training
VAL_STEPS = 30  # inference steps
VAL_LOSS_IMAGES = 4  # held-out paintings of every artist for the validation loss
VAL_LOSS_TIMESTEPS = 5  # fixed timesteps of the validation loss
VAL_GUIDANCE = 7.5
NUM_HOLDOUT = 1  # captions of the held-out paintings of every artist added to the prompts
VAL_PROMPTS = [  # {trigger} -> one prompt per artist
    '{trigger}, a river valley with a forest at sunset',
    '{trigger}, a portrait of an old man with a beard',
    'a river valley with a forest at sunset',  # no trigger: the style must not leak
]

RUN_NAME = f'sd15_lora_{datetime.now():%Y%m%d-%H%M%S}'

hparams = {k: v for k, v in globals().items() if k not in _defined and k != '_defined'}

In [ ]:
log_dir = ROOT / 'logs/art-style-adapters' / RUN_NAME
ckpt_dir = log_dir / 'checkpoints'
eval_dir = log_dir / 'eval'
ckpt_dir.mkdir(parents=True, exist_ok=True)
write_json(hparams, log_dir / 'params.json')
print(f'Run directory: {log_dir}')

In [ ]:
set_seed(SEED)
accelerator = Accelerator(
    gradient_accumulation_steps=GRAD_ACCUM,
    mixed_precision=get_mixed_precision(MIXED_PRECISION),
)
device = accelerator.device
print(f'device={device}, mixed_precision={accelerator.mixed_precision}')

# frozen SD + trainable adapter on the UNet
sd = load_sd(MODEL, device, accelerator.mixed_precision)
peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    init_lora_weights='gaussian',
    target_modules=TARGET_MODULES,
)
attach_adapter(sd.unet, peft_config, GRADIENT_CHECKPOINTING)
print(get_parameter_summary(sd.unet))

params = [p for p in sd.unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

train_set = StyleDataset(
    path=DATA_DIR,
    artists=TRAIN_ARTISTS,
    tokenizer=sd.tokenizer,
    split='train',
    resolution=RESOLUTION,
    center_crop=CENTER_CROP,
    random_flip=RANDOM_FLIP,
)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

steps_per_epoch = get_steps_per_epoch(train_loader, GRAD_ACCUM, accelerator.num_processes)
num_steps = NUM_EPOCHS * steps_per_epoch
print(f'{steps_per_epoch} steps per epoch, {num_steps} steps')
lr_scheduler = get_scheduler(
    'cosine',
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS * accelerator.num_processes,
    num_training_steps=num_steps * accelerator.num_processes,
)

sd.unet, optimizer, train_loader, lr_scheduler = accelerator.prepare(
    sd.unet, optimizer, train_loader, lr_scheduler
)

In [ ]:
# the pipeline shares the weights with the unet that is trained
holdout = load_artist_split(DATA_DIR, TRAIN_ARTISTS, 'test')
pipe = build_pipeline(sd, accelerator.unwrap_model(sd.unet), device)
val_prompts = get_val_prompts(VAL_PROMPTS, holdout, NUM_HOLDOUT, eval_dir)
val_metrics = ValMetrics(
    sd, accelerator, DATA_DIR, TRAIN_ARTISTS, RESOLUTION, VAL_LOSS_IMAGES, VAL_LOSS_TIMESTEPS, seed=SEED
)


def eval_fn(step: int) -> None:
    metrics = val_metrics()
    print(
        f'step {step}: val_loss={metrics["val/loss"]:.4f} '
        f'(gain {metrics["val/gain"]:+.4f}), without trigger '
        f'{metrics["val/loss_no_trigger"]:.4f} (gain {metrics["val/gain_no_trigger"]:+.4f}), '
        f'adapter delta {metrics["adapter/delta_norm"]:.3f}'
    )
    generate_samples(
        pipe, accelerator, val_prompts, eval_dir, step, VAL_STEPS, VAL_GUIDANCE, SEED
    )


def save_fn(step: int) -> None:
    save_adapter(accelerator.unwrap_model(sd.unet), peft_config, ckpt_dir / f'checkpoint-{step:04d}')


# the adapter is zero-initialized (LoRA B matrices), so step 0 is the base model
eval_fn(0)

In [ ]:
train(
    accelerator=accelerator,
    sd=sd,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    train_loader=train_loader,
    num_epochs=NUM_EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
    log_step=LOG_STEP,
    eval_every=VAL_EVERY,
    eval_fn=eval_fn,
    save_every=SAVE_EVERY,
    save_fn=save_fn,
)
print(f'Done: adapters in {ckpt_dir}, samples in {eval_dir}')

## 4. Results

The samples are generated before the training (step 0 is the base model), every `VAL_EVERY` steps and at the end. The prompts are in `eval/prompts.json`, the held-out paintings for the comparison are in `eval/holdout`.

In [ ]:
prompts = read_json(eval_dir / 'prompts.json')
for i, prompt in enumerate(prompts):
    print(f'{i:02d}: {prompt}')


def show_step(step_dir: Path, cols: int = 6, size: int = 192) -> None:
    """Contact sheet of the samples of one step, the order is the order of the prompts."""
    images = [np.array(Image.open(f).convert('RGB').resize((size, size))) for f in sorted(step_dir.glob('*.png'))]
    images += [np.zeros_like(images[0])] * (-len(images) % cols)
    rows = [np.hstack(images[i : i + cols]) for i in range(0, len(images), cols)]
    print(step_dir.name)
    display(Image.fromarray(np.vstack(rows)))


steps = sorted(eval_dir.glob('step_*'))
show_step(steps[0])  # the base model
show_step(steps[-1])  # the trained adapter